# 00 — Setup y verificación del entorno

**Objetivo:** confirmar que las librerías, los datos y MLflow funcionan antes de empezar el análisis.

Si este notebook corre de punta a punta sin errores, el entorno está listo.

## 1. Importar el módulo de configuración

Los notebooks viven en `notebooks/`, pero `config.py` está en `src/`. Agregamos la raíz del repo al `path` de Python para poder importarlo.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import config
print("Raíz del proyecto:", config.ROOT)

Raíz del proyecto: /Users/milagrossachetti/Downloads/parcial-moscow


## 2. Librerías y versiones

Registrar las versiones permite que cualquiera reproduzca exactamente los resultados.

In [2]:
import pandas as pd, numpy as np, matplotlib, seaborn as sns
import sklearn, statsmodels, mlflow

for lib in [pd, np, matplotlib, sns, sklearn, statsmodels, mlflow]:
    print(f"{lib.__name__:<12} {lib.__version__}")

2026/10/17 19:46:11 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/milagrossachetti/Downloads/parcial-moscow/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


pandas       3.0.5
numpy        2.5.3
matplotlib   3.11.2
seaborn      0.13.2
sklearn      1.9.1
statsmodels  0.15.0
mlflow       3.17.1


## 3. Carga de datos

Solo verificamos que el archivo se lee y tiene la forma esperada. El análisis va en el notebook 01.

In [3]:
df = pd.read_csv(config.SECONDARY_CSV, parse_dates=["date_posted"])
print("Filas x columnas:", df.shape)
print("Período:", df["date_posted"].min().date(), "→", df["date_posted"].max().date())
df.head()

Filas x columnas: (50000, 26)
Período: 2020-01-01 → 2026-04-29


,id,date_posted,district,okrug,lat,lon,total_area,living_area,kitchen_area,rooms,...,renovation,metro_station,metro_line,metro_distance_min,metro_distance_type,to_center_km,price_rub,price_per_sqm,mortgage_rate_at_listing,seller_type
0,S000001,2024-11-23,Begovoy,SAO,55.79533,37.55519,71.0,46.5,6.1,2,...,cosmetic,Sokol,Zamoskvoretskaya,18,transport,6.17,15739000,221700,23.5,agency
1,S000002,2025-08-13,Sokolniki,VAO,55.79695,37.68875,79.6,52.8,10.8,3,...,no_renovation,Sokolniki,Sokolnicheskaya,7,walk,6.40,19685000,247300,19.5,owner
2,S000003,2022-05-06,Sosenskoye,NAO,55.54901,37.43118,74.0,42.2,8.9,2,...,cosmetic,Polyanka,Serpukhovsko-Timiryazevskaya,102,transport,25.70,8621000,116400,12.0,owner
3,S000004,2025-04-08,Sviblovo,SVAO,55.86462,37.61471,52.3,34.0,6.0,1,...,euro,Sviblovo,Kaluzhsko-Rizhskaya,9,walk,12.32,14771000,282400,23.5,agency
4,S000005,2023-07-16,Begovoy,SAO,55.79429,37.54631,62.2,38.3,13.1,2,...,cosmetic,Sapyornoye Pole,Bolshaya Koltsevaya,10,walk,6.47,18683000,300300,15.5,agency


In [4]:
# El target existe y las columnas excluidas están identificadas
assert config.TARGET in df.columns
print("Target:", config.TARGET)
print("Excluidas del modelo:", config.EXCLUDED_COLS)

Target: price_per_sqm
Excluidas del modelo: ['id', 'price_rub']


## 4. Conexión con MLflow

`setup_mlflow()` apunta MLflow al archivo `mlflow.db` del repo y crea el experimento si no existe.
Acá **no** registramos ninguna ejecución: el primer *run* real será el modelo base (baseline).

In [5]:
mlflow = config.setup_mlflow()
exp = mlflow.get_experiment_by_name(config.MLFLOW_EXPERIMENT)
print("Tracking URI :", mlflow.get_tracking_uri())
print("Experimento  :", exp.name, "| id:", exp.experiment_id)
print("Artefactos en:", exp.artifact_location)

Tracking URI : sqlite:////Users/milagrossachetti/Downloads/parcial-moscow/mlflow.db
Experimento  : moscow-secondary-price-per-sqm | id: 1
Artefactos en: mlartifacts


✅ Entorno listo. Siguiente: `01_eda.ipynb`.